# Fix $\mathrm{tasmin} \gt \mathrm{tasmax}$ issue

I got in touch with NEX to clarify the issue in NEX-GDDP-CMIP6 dataset (cf. file `debug_tas_extremes.ipynb`).
Until then, the temporary fix I implement is to swap the extreme values to restore the coherence in those points which show the issue. Given the low relative number of non-physical occurences, this should not affect statistically the distribution of tasmin and tasmax (even though I still have to think how to prove it).

For instance, if for a given day and grid point the values are

| tasmin | tasmax |
|--------|--------|
| 26 °C  | 16 °C  |

hence $\mathrm{tasmax} \leq \mathrm{tasmin}$, then I reorder these values and reassign the variables names to keep coherence with their meaning (i.e. $\mathrm{tasmin} \leq \mathrm{tasmax}$), resulting in

| tasmin | tasmax |
|--------|--------|
| 16 °C  | 34 °C  |

After this correction, the bias adjustment is applied to tasmax and the DTR evaluated on fixed data, as suggested for [previous instances of the NEX datasets](https://doi.org/10.5194/hess-16-3309-2012).

I apply these changes only to tasmin and tasmax variables (even if I can show that also tas values are not coherent with the extremes) because the indicators I consider may depend only on these variables simultaneously.

In [1]:
import importlib

import xarray

import settings
import thesis

# Workaround to reload settings if this block is executed a second time but notebook is not restarted.
try:
    id(flag_reload)
    flag_reload = importlib.reload(settings)
except NameError:  # Raised if `flag_reload` is not defined, hence when block is executed the first time.
    flag_reload = settings

In [2]:
dir_data = settings.dir_data / settings.location / "preprocessed"

def swap_tasmin_tasmax(ds):
    dtr = ds["tasmax"] - ds["tasmin"]  # Work on full dataset without creating copies.
    idx_right = (dtr >= 0).compute()
    tasmax_old = ds["tasmax"].copy(deep=True)
    ds = ds.assign({
        "tasmax": ds["tasmax"].where(idx_right, other=ds["tasmin"]),
        "tasmin": ds["tasmin"].where(idx_right, other=tasmax_old)
    })
    idx_wrong = (idx_right == False)  # Return sequence of models with tasmin > tasmax.
    return (ds, tuple(idx_wrong.where(idx_wrong, drop=True)["model"].values))

In [3]:
chunks = {"latitude": 1, "longitude": 1, "time": -1}
m_dim = "model"
s_dim = "scenario"

dict_wrong = {}

with xarray.open_mfdataset(
    [dir_data / f"historical_{m}.nc" for m in thesis.models],
    chunks=chunks,
    concat_dim=m_dim,
    combine="nested",
    combine_attrs="drop_conflicts"
) as _ds:
    _ds = _ds.chunk({m_dim: -1})
    _ds = _ds.assign_coords({m_dim: list(thesis.models)})
    dict_wrong["historical"] = _ds.persist()

for s in thesis.scenario:
    with xarray.open_mfdataset(
        [dir_data / f"{s}_{m}.nc" for m in thesis.models],
        chunks=chunks,
        concat_dim=m_dim,
        combine="nested",
        combine_attrs="drop_conflicts"
    ) as _ds:
        _ds = _ds.chunk({m_dim: -1})
        _ds = _ds.assign_coords({m_dim: list(thesis.models)})
        dict_wrong[s] = _ds.persist()

In [4]:
dict_right = {}
dict_wrong_models = {}
for e in dict_wrong.keys():
    _return = swap_tasmin_tasmax(dict_wrong[e])
    dict_right[e] = _return[0]
    dict_wrong_models[e] = _return[1]

In [5]:
dir_bak = dir_data / "bak_tas_extremes"
dir_bak.mkdir(parents=True, exist_ok=True)

for e in dict_wrong.keys():
    for m in dict_wrong_models[e]:
        filename = f"{e}_{m}.nc"
        
        # Backup preprocessed file for future reference.
        path = dir_data / filename
        path.replace(dir_bak / filename)
        
        # Store fixed preprocessed file.
        dict_right[e].sel(model=m, drop=True).to_netcdf(path)